In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
path = kagglehub.dataset_download("abhinavwalia95/entity-annotated-corpus")

print("Dataset Path:", path)

100%|██████████| 26.4M/26.4M [00:00<00:00, 90.6MB/s]

Extracting files...


Dataset Path: /root/.cache/kagglehub/datasets/abhinavwalia95/entity-annotated-corpus/versions/4


In [ ]:
file_path = os.path.join(path, "ner_dataset.csv")

df = pd.read_csv(file_path, encoding="latin1")

df = df.fillna(method="ffill")

df.head()

/tmp/ipykernel_376/3499585800.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")


,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O


In [ ]:
sentences = df.groupby("Sentence #")["Word"].apply(list).values
tags = df.groupby("Sentence #")["Tag"].apply(list).values

print("Example Sentence:", sentences[0])
print("Example Tags:", tags[0])

Example Sentence: ['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.']
Example Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O']


In [ ]:
words = list(set(df["Word"].values))
tags_unique = list(set(df["Tag"].values))

word2idx = {w:i+1 for i,w in enumerate(words)}
tag2idx = {t:i for i,t in enumerate(tags_unique)}

idx2tag = {i:t for t,i in tag2idx.items()}

vocab_size = len(word2idx) + 1
num_tags = len(tag2idx)

print("Vocabulary size:", vocab_size)
print("Number of tags:", num_tags)

Vocabulary size: 35178
Number of tags: 17


In [ ]:
max_len = 30

def encode_data(sentences, tags):

    X = []
    y = []

    for sent, tag in zip(sentences, tags):

        word_ids = [word2idx[w] for w in sent]
        tag_ids = [tag2idx[t] for t in tag]

        word_ids = word_ids[:max_len]
        tag_ids = tag_ids[:max_len]

        padding = max_len - len(word_ids)

        word_ids += [0]*padding
        tag_ids += [0]*padding

        X.append(word_ids)
        y.append(tag_ids)

    return np.array(X), np.array(y)


X, y = encode_data(sentences, tags)

print(X.shape)

(47959, 30)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
class NERDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = NERDataset(X_train, y_train)
test_dataset = NERDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
class BiLSTM_NER(nn.Module):

    def __init__(self, vocab_size, num_tags):

        super(BiLSTM_NER, self).__init__()

        self.embedding = nn.Embedding(vocab_size, 128)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256, num_tags)

    def forward(self, x):

        x = self.embedding(x)

        lstm_out, _ = self.lstm(x)

        output = self.fc(lstm_out)

        return output


model = BiLSTM_NER(vocab_size, num_tags)

print(model)

BiLSTM_NER(
  (embedding): Embedding(35178, 128)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=17, bias=True)
)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 3

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        outputs = outputs.view(-1, num_tags)
        y_batch = y_batch.view(-1)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss)

Epoch: 1 Loss: 288.4924330972135
Epoch: 2 Loss: 115.08825988695025
Epoch: 3 Loss: 81.75833768211305


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        predictions = torch.argmax(outputs, dim=2)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.numel()

accuracy = correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 0.974391854323047


In [ ]:
sample = sentences[5]

encoded = [word2idx.get(w,0) for w in sample]

encoded = encoded[:max_len] + [0]*(max_len-len(encoded))

input_tensor = torch.tensor([encoded])

with torch.no_grad():

    output = model(input_tensor)

    prediction = torch.argmax(output, dim=2)

pred_tags = [idx2tag[i.item()] for i in prediction[0][:len(sample)]]

print("Sentence:", sample)
print("Predicted Tags:", pred_tags)

Sentence: ['Mr.', 'Egeland', 'said', 'the', 'latest', 'figures', 'show', '1.8', 'million', 'people', 'are', 'in', 'need', 'of', 'food', 'assistance', '-', 'with', 'the', 'need', 'greatest', 'in', 'Indonesia', ',', 'Sri', 'Lanka', ',', 'the', 'Maldives', 'and', 'India', '.']
Predicted Tags: ['B-per', 'I-per', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'B-geo', 'I-geo', 'O', 'O', 'B-geo', 'O']


In [ ]:
# Function to predict tags for a custom sentence

def predict_sentence(sentence):

    model.eval()

    words = sentence.split()

    # Convert words to index
    encoded = [word2idx.get(w, 0) for w in words]

    # Padding
    encoded = encoded[:max_len] + [0]*(max_len-len(encoded))

    input_tensor = torch.tensor([encoded])

    with torch.no_grad():

        output = model(input_tensor)

        prediction = torch.argmax(output, dim=2)

    pred_tags = [idx2tag[i.item()] for i in prediction[0][:len(words)]]

    # Print result
    print("\nSentence:", sentence)
    print("\nWord  ->  Predicted Tag")

    for w, t in zip(words, pred_tags):
        print(f"{w}  ->  {t}")


# Example sentence
predict_sentence("Barack Obama visited India")


Sentence: Barack Obama visited India

Word  ->  Predicted Tag
Barack  ->  B-per
Obama  ->  I-per
visited  ->  O
India  ->  B-geo


In [ ]:
predict_sentence("I Love Natural Language Processing.")


Sentence: I Love Natural Language Processing.

Word  ->  Predicted Tag
I  ->  O
Love  ->  B-per
Natural  ->  I-per
Language  ->  I-gpe
Processing.  ->  I-gpe
